# End-to-End Stock Market Forecasting with LSTM & Technical Indicators

This notebook develops a state-of-the-art deep learning pipeline for forecasting stock market price trajectories using historical market data, advanced technical indicators, leakage-free preprocessing, and LSTM neural networks.

### Key Highlights & Upgrades:
- **Stationary Feature & Target Formulation:** Overcomes the common random-walk persistence pitfall by predicting relative price returns rather than non-stationary raw prices.
- **Multivariate Technical Indicators:** Integrates RSI (14), MACD (12, 26, 9), Moving Average Ratios (MA10, MA20, MA50), High-Low Spread, and Volume Momentum.
- **Strict Leakage-Free Scaling:** Fits StandardScaler strictly on training data; validation and test sets are transformed without leakage.
- **Modernized Architecture:** 2-layer LSTM with canonical tanh/sigmoid activations, Dropout regularization, EarlyStopping, and ReduceLROnPlateau.
- **Comprehensive Evaluation:** Computes MAE, RMSE, MAPE, R², and Directional Accuracy (%) alongside Naive Persistence and Moving Average baselines.
- **Multi-Step Future Forecasting:** Generates autoregressive multi-day price projections beyond historical records.


## 1. Imports and Environment Setup


In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import json
import joblib
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam


## 2. Historical Data Collection (GOOG)


In [ ]:
stock = 'GOOG'
start = '2012-01-01'
end = '2022-12-21'

raw_data = yf.download(stock, start=start, end=end, progress=False)
raw_data.dropna(inplace=True)

if isinstance(raw_data.columns, pd.MultiIndex):
    close = raw_data['Close'][stock] if stock in raw_data['Close'] else raw_data['Close'].iloc[:, 0]
    high = raw_data['High'][stock] if stock in raw_data['High'] else raw_data['High'].iloc[:, 0]
    low = raw_data['Low'][stock] if stock in raw_data['Low'] else raw_data['Low'].iloc[:, 0]
    open_p = raw_data['Open'][stock] if stock in raw_data['Open'] else raw_data['Open'].iloc[:, 0]
    volume = raw_data['Volume'][stock] if stock in raw_data['Volume'] else raw_data['Volume'].iloc[:, 0]
else:
    close = raw_data['Close']
    high = raw_data['High']
    low = raw_data['Low']
    open_p = raw_data['Open']
    volume = raw_data['Volume']

df = pd.DataFrame(index=raw_data.index)
df['Open'] = open_p
df['High'] = high
df['Low'] = low
df['Close'] = close
df['Volume'] = volume

print(f'Total records downloaded: {len(df)}')
df.head()


## 3. Multivariate Technical Feature Engineering


In [ ]:
# 1. Stationary Relative Return & Spread
df['Return'] = df['Close'].pct_change()
df['HL_Spread'] = (df['High'] - df['Low']) / df['Close']

# 2. Moving Average Ratios
df['MA10'] = df['Close'].rolling(10).mean()
df['MA20'] = df['Close'].rolling(20).mean()
df['MA50'] = df['Close'].rolling(50).mean()

df['MA10_Ratio'] = df['Close'] / df['MA10'] - 1.0
df['MA20_Ratio'] = df['Close'] / df['MA20'] - 1.0
df['MA50_Ratio'] = df['Close'] / df['MA50'] - 1.0

# 3. Relative Strength Index (RSI 14)
delta = df['Close'].diff()
gain = (delta.where(delta > 0, 0)).rolling(14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
rs = gain / (loss + 1e-9)
df['RSI'] = 100.0 - (100.0 / (1.0 + rs))
df['RSI_Scaled'] = df['RSI'] / 100.0 - 0.5  # centered to [-0.5, 0.5]

# 4. Moving Average Convergence Divergence (MACD 12, 26, 9)
ema12 = df['Close'].ewm(span=12, adjust=False).mean()
ema26 = df['Close'].ewm(span=26, adjust=False).mean()
df['MACD'] = ema12 - ema26
df['MACD_Signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
df['MACD_Ratio'] = df['MACD'] / df['Close']

# 5. Volume Change
df['Vol_Change'] = df['Volume'].pct_change().clip(-1.0, 3.0)

df.dropna(inplace=True)
print(f'Engineered dataset shape: {df.shape}')


## 4. Leakage-Free Preprocessing & Sequence Creation


In [ ]:
feature_cols = ['Return', 'HL_Spread', 'MA10_Ratio', 'MA20_Ratio', 'MA50_Ratio', 'RSI_Scaled', 'MACD_Ratio', 'Vol_Change']
seq_len = 30

features = df[feature_cols].values
returns = df['Return'].values
prices = df['Close'].values
dates = df.index

X, y_ret, p_prev, p_curr, seq_dates = [], [], [], [], []
for i in range(seq_len, len(df)):
    X.append(features[i-seq_len:i])
    y_ret.append(returns[i])
    p_prev.append(prices[i-1])
    p_curr.append(prices[i])
    seq_dates.append(dates[i])

X = np.array(X)
y_ret = np.array(y_ret)
p_prev = np.array(p_prev)
p_curr = np.array(p_curr)

# 80/20 Chronological Split
split = int(len(X) * 0.80)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y_ret[:split], y_ret[split:]
p_prev_train, p_prev_test = p_prev[:split], p_prev[split:]
p_curr_train, p_curr_test = p_curr[:split], p_curr[split:]
test_dates = seq_dates[split:]

# Zero-Leakage StandardScaler fitted on train set ONLY
scaler = StandardScaler()
N_tr, T, D = X_train.shape
X_train_scaled = scaler.fit_transform(X_train.reshape(-1, D)).reshape(N_tr, T, D)

N_te = X_test.shape[0]
X_test_scaled = scaler.transform(X_test.reshape(-1, D)).reshape(N_te, T, D)

print(f'Train samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}')


## 5. Model Architecture & Training


In [ ]:
model = Sequential([
    Input(shape=(T, D)),
    LSTM(48, activation='tanh', recurrent_activation='sigmoid', return_sequences=True),
    Dropout(0.2),
    LSTM(24, activation='tanh', recurrent_activation='sigmoid', return_sequences=False),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1)
])

model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error')
model.summary()

callbacks = [
    EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, verbose=1)
]

history = model.fit(
    X_train_scaled, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.15,
    callbacks=callbacks,
    verbose=1
)


## 6. Comprehensive Evaluation & Baseline Comparison


In [ ]:
# Predict on Test Set
pred_returns = model.predict(X_test_scaled).flatten()
pred_prices = p_prev_test * (1.0 + pred_returns)
actual_prices = p_curr_test

# Compute Metrics
mae = mean_absolute_error(actual_prices, pred_prices)
rmse = np.sqrt(mean_squared_error(actual_prices, pred_prices))
mape = np.mean(np.abs((actual_prices - pred_prices) / actual_prices)) * 100.0
r2 = r2_score(actual_prices, pred_prices)
da = np.mean(np.sign(actual_prices - p_prev_test) == np.sign(pred_prices - p_prev_test)) * 100.0

# Baseline 1: Naive Persistence (tomorrow = today)
mae_naive = mean_absolute_error(actual_prices, p_prev_test)
rmse_naive = np.sqrt(mean_squared_error(actual_prices, p_prev_test))
mape_naive = np.mean(np.abs((actual_prices - p_prev_test) / actual_prices)) * 100.0
r2_naive = r2_score(actual_prices, p_prev_test)

# Baseline 2: 20-day Simple Moving Average
ma20_test = df['MA20'].iloc[-len(actual_prices):].values
mae_ma = mean_absolute_error(actual_prices, ma20_test)
rmse_ma = np.sqrt(mean_squared_error(actual_prices, ma20_test))
mape_ma = np.mean(np.abs((actual_prices - ma20_test) / actual_prices)) * 100.0
r2_ma = r2_score(actual_prices, ma20_test)

results_df = pd.DataFrame({
    'Model / Baseline': ['Enhanced LSTM', 'Naive Persistence', '20-Day SMA'],
    'MAE ($)': [mae, mae_naive, mae_ma],
    'RMSE ($)': [rmse, rmse_naive, rmse_ma],
    'MAPE (%)': [mape, mape_naive, mape_ma],
    'R2 Score': [r2, r2_naive, r2_ma],
    'Directional Acc (%)': [da, 50.0, 50.0]
})

print(results_df.to_string(index=False))


## 7. Prediction Visualizations


In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(test_dates, actual_prices, color='green', label='Actual Price', linewidth=2)
plt.plot(test_dates, pred_prices, color='red', linestyle='--', label='LSTM Predicted Price', linewidth=1.8)
plt.title(f'{stock} Actual Price vs. LSTM Predicted Price (Test Set)', fontsize=14)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Stock Price ($)', fontsize=12)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 8. Multi-Step Future Forecasting


In [ ]:
def compute_dynamic_indicators(df):
    data = df.copy()
    data['Return'] = data['Close'].pct_change()
    if 'High' in data.columns and 'Low' in data.columns:
        data['HL_Spread'] = (data['High'] - data['Low']) / data['Close']
    else:
        data['HL_Spread'] = 0.01
        
    data['MA10_Ratio'] = data['Close'] / data['Close'].rolling(10).mean() - 1.0
    data['MA20_Ratio'] = data['Close'] / data['Close'].rolling(20).mean() - 1.0
    data['MA50_Ratio'] = data['Close'] / data['Close'].rolling(50).mean() - 1.0
    
    delta = data['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
    rs = gain / (loss + 1e-9)
    data['RSI'] = 100.0 - (100.0 / (1.0 + rs))
    data['RSI_Scaled'] = data['RSI'] / 100.0 - 0.5
    
    ema12 = data['Close'].ewm(span=12, adjust=False).mean()
    ema26 = data['Close'].ewm(span=26, adjust=False).mean()
    data['MACD'] = ema12 - ema26
    data['MACD_Ratio'] = data['MACD'] / data['Close']
    
    if 'Volume' in data.columns:
        data['Vol_Change'] = data['Volume'].pct_change().clip(-1.0, 3.0)
    else:
        data['Vol_Change'] = 0.0
    return data

def forecast_next_days(model, scaler, df, feature_cols, days=14, seq_len=30):
    curr_df = df.copy()
    for col in ['Open', 'High', 'Low', 'Close']:
        if col not in curr_df.columns:
            curr_df[col] = curr_df['Close'] if 'Close' in curr_df.columns else 100.0
    if 'Volume' not in curr_df.columns:
        curr_df['Volume'] = 1000000
        
    future_preds = []
    last_date = curr_df.index[-1]
    
    for _ in range(days):
        seq = curr_df[feature_cols].iloc[-seq_len:].values
        seq_sc = scaler.transform(seq).reshape(1, seq_len, len(feature_cols))
        ret = float(model.predict(seq_sc, verbose=0)[0, 0])
        
        last_close = float(curr_df['Close'].iloc[-1])
        new_close = last_close * (1.0 + ret)
        
        last_date += datetime.timedelta(days=1)
        while last_date.weekday() >= 5:
            last_date += datetime.timedelta(days=1)
            
        future_preds.append((last_date, new_close))
        
        new_row = pd.DataFrame([{
            'Open': new_close, 'High': new_close * 1.005, 'Low': new_close * 0.995,
            'Close': new_close, 'Volume': curr_df['Volume'].iloc[-1]
        }], index=[last_date])
        
        extended = pd.concat([curr_df[['Open', 'High', 'Low', 'Close', 'Volume']], new_row])
        curr_df = compute_dynamic_indicators(extended)
        
    return pd.DataFrame(future_preds, columns=['Date', 'Forecast_Price']).set_index('Date')

fc_df = forecast_next_days(model, scaler, df, feature_cols, days=14)
print('Projected Next 14 Trading Days:')
print(fc_df.head(10))


## 9. Save Model & Scaler Artifacts


In [ ]:
model.save('Stock Predictions Model.keras')
joblib.dump(scaler, 'feature_scaler.joblib')

metrics_dict = {
    'symbol': stock,
    'records_total': len(df),
    'records_train': len(X_train),
    'records_test': len(X_test),
    'lstm': {'mae': mae, 'rmse': rmse, 'mape': mape, 'r2': r2, 'directional_accuracy': da},
    'naive_baseline': {'mae': mae_naive, 'rmse': rmse_naive, 'mape': mape_naive, 'r2': r2_naive},
    'ma20_baseline': {'mae': mae_ma, 'rmse': rmse_ma, 'mape': mape_ma, 'r2': r2_ma}
}

with open('metrics.json', 'w') as f:
    json.dump(metrics_dict, f, indent=4)

print('All artifacts successfully saved.')
